# Model training

Trains and — more importantly — honestly evaluates a per-pixel classifier for waterhole
surface state.

## Read this before reading any score

**Your effective sample size is the number of labelled sites, not pixels.** The table below
has ~32,000 rows, but pixels within a tile are almost perfectly autocorrelated: knowing one
tells you most of what its neighbours will say. With 9 labelled sites, the honest n is 9.

Every split in this notebook therefore holds out **whole waterholes**. `wh_train` has no
code path for a random pixel-level split — it does not import `train_test_split` or `KFold`
at all, and its only splitter raises if handed fewer than 3 sites. A random split here would
report something like 0.95 and mean nothing.

Expect the flexible models to do *worse* than the simple one. That is not a bug; it is what
overfitting to 9 sites looks like.

In [ ]:
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == "cookie-cutting" else Path.cwd() / "cookie-cutting"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import wh_config
import wh_features
import wh_inventory
import wh_plots
import wh_tiles
import wh_train

pd.set_option("display.width", 140)

cfg = wh_config.load()
manifest = wh_inventory.load_manifest(cfg)
print("config", cfg.source_path.name, "hash", cfg.hash)

## Parameters

In [ ]:
PARAMS = wh_features.FeatureParams(
    # 60 m atmospheric bands B1 and B9 excluded: resampled to 10 m they carry no
    # surface information worth having.
    reflectance_bands=("B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"),

    indices=("mndwi", "ndwi", "ndvi", "ndti", "ndmi", "nd_rededge",
             "red_green_ratio", "awei_sh", "awei_nsh"),

    # Windowed mean and SD, so a pixel knows whether it sits in a uniform patch
    # or on an edge. NaN-aware: a window overlapping a cloud gap averages the
    # pixels that were observed, not zeros.
    context_windows=(3, 9),
    context_indices=("mndwi", "ndvi"),

    # Indices given the full temporal treatment. NDTI carries turbidity and NDMI
    # responds to water UNDER a canopy — the case MNDWI gets wrong when sedges
    # cover a waterhole.
    temporal_indices=("mndwi", "ndvi", "ndti", "ndmi"),
)

INCLUDE_PSEUDO = False    # run pseudo_labelling.ipynb first, then try True
REBUILD_TABLE = True      # False reloads the cached table from derived/

PARAMS

## Build the training table

One row per labelled pixel, carrying `site_id` and `year_month` alongside every feature.
Those two columns are what make a grouped split possible, so they travel with every row.

Sites are processed one at a time so each site's temporal features are computed once rather
than once per labelled month. Expect a couple of minutes.

In [ ]:
if REBUILD_TABLE:
    table = wh_train.build_training_table(manifest, cfg, PARAMS, include_pseudo=INCLUDE_PSEUDO)
    wh_train.save_table(table, cfg)
else:
    table = wh_train.load_table(cfg)

features = wh_features.feature_columns(table)
print(f"\n{len(table):,} rows x {len(features)} features")
print(f"{table['site_id'].nunique()} sites, {table['year_month'].nunique()} months")
print(f"sources: {table['source'].value_counts().to_dict()}")

### Class coverage — the table that predicts what will fail

Read the **sites** column, not the pixels column. A class present at only 2 sites will be
absent from training entirely in some folds, and cannot be expected to work.

In [ ]:
coverage = table.groupby("class_id").agg(
    pixels=("row", "size"),
    tiles=("year_month", "nunique"),
    sites=("site_id", "nunique"),
)
coverage.insert(0, "class", [cfg.class_by_id(i).name for i in coverage.index])
coverage = coverage.sort_values("sites")

thin = coverage[coverage["sites"] < 3]
if len(thin):
    print("classes at fewer than 3 sites — expect these to fail, and not because "
          "the model is bad:")
    print("  " + ", ".join(thin["class"]))
coverage

In [ ]:
# Features that are mostly NaN mean too few pixels had enough observed months
# to fit — a data problem, not something for the imputer to paper over.
missing = wh_features.describe_missing(table, threshold=0.2)
print(f"{len(missing)} feature(s) more than 20% NaN")
missing.head(10)

## Cross-validation

Leave-one-site-out: 9 folds, each holding out one whole waterhole. The per-fold line shows
which sites the model can and cannot generalise to.

In [ ]:
results = {}
for model_name in ("logistic_regression", "random_forest", "gradient_boosting"):
    print(f"\n=== {model_name} ===")
    results[model_name] = wh_train.cross_validate(table, cfg, model_name)
    print(results[model_name].summary())

print("\n" + "=" * 60)
pd.DataFrame([
    {"model": name, "macro_f1": ev.macro_f1, "weighted_f1": ev.weighted_f1}
    for name, ev in results.items()
]).set_index("model").round(3)

**Logistic regression is a diagnostic, not a contender.**

With 9 labelled sites, the flexible models have far more capacity than the data supports and
overfit the sites they were given. If a linear model on the same features generalises better
to a held-out waterhole, that is the signal to add *labelled sites*, not more features or a
bigger model.

The best model is therefore chosen from the scores below rather than fixed in advance — the
ranking will shift as you label more, and it should.

In [ ]:
# Chosen on cross-validated evidence, not decided in advance.
BEST = max(results, key=lambda name: results[name].macro_f1)

print("leave-one-site-out macro F1:")
for name, ev in sorted(results.items(), key=lambda kv: -kv[1].macro_f1):
    marker = "  <- selected" if name == BEST else ""
    print(f"  {name:22s} {ev.macro_f1:.3f}{marker}")

# BEST = "gradient_boosting"    # uncomment to override the choice
evaluation = results[BEST]

print(f"\nper class ({BEST}):")
print(evaluation.per_class.round(3).to_string())
print("\nmacro F1 weights every class equally, so the rare broken ones drag it down;")
print("weighted F1 is dominated by surrounding vegetation and dry bare ground.")

In [ ]:
print("per site (the table that matters):")
evaluation.per_site.round(3)

### Diagnostic plots

The confusion matrix normalised by true class — "of the pixels that really were mud, where
did they go?" — with raw counts in each cell so the support behind each rate stays visible.
A 100% rate over 12 pixels is not the same claim as 100% over 12,000.

In [ ]:
figure = wh_plots.plot_confusion(evaluation, normalise="true")
plt.show()

figure = wh_plots.plot_class_scores(evaluation)
plt.show()

figure = wh_plots.plot_site_scores(evaluation)
plt.show()

## Ablation: does the temporal design earn its keep?

The central claim of this pipeline is that normalising a pixel against its own history makes
the ambiguous dry-season months separable. `instantaneous_only` tests that directly — it is
the same model with the temporal columns removed.

**The answer depends on the model, and that itself is the finding.** Same leave-one-site-out
splits:

| feature set | gradient boosting | logistic regression |
|---|---|---|
| instantaneous_only (28) | 0.358 | 0.512 |
| with temporal (57) | 0.497 | 0.540 |
| **gain** | **+0.140** | **+0.028** |

Logistic regression on instantaneous features *alone* beats gradient boosting with
everything. So most of the apparent +0.14 was the temporal block compensating for the tree
model overfitting 9 sites, not new information. The temporal features do still help — but by
+0.03 on the model that actually generalises, not by +0.14.

**Rows marked `identical_to` were not re-run.** With `harmonic_enabled: false` the harmonic
and trend blocks are empty, so `all_features`, `no_harmonic` and `model_free_temporal` are
the same 57 columns. They are scored once and the result carried across, rather than
cross-validated three times to produce three identical numbers. Set `harmonic_enabled: true`
and rebuild the table to get a live comparison against the harmonic block.

In [ ]:
ablation = wh_train.run_ablation(table, cfg, PARAMS, model_name=BEST)
print()
ablation.round(4)

figure = wh_plots.plot_ablation(ablation)
plt.show()

## Temporal holdout

Reported for completeness and **known to be optimistically biased**: a held-out month's
temporal features were computed from that pixel's whole history, training months included.
Removing that leak would mean recomputing every temporal feature per fold from training
months only.

The grouped-site CV above has no equivalent problem — holding out a site holds out its
history too. Trust that number, not this one.

In [ ]:
train_index, test_index = wh_train.temporal_holdout_split(
    table, cfg["training"]["cv"]["temporal_holdout_months"]
)
model = wh_train.make_model(BEST, cfg)
model.fit(table.iloc[train_index][features], table.iloc[train_index]["class_id"])
predicted = model.predict(table.iloc[test_index][features])

holdout = wh_train.evaluate_predictions(
    table.iloc[test_index]["class_id"].to_numpy(), predicted,
    table.iloc[test_index]["site_id"].to_numpy(), cfg, BEST, "temporal_holdout",
)
print(holdout.summary(), "  <- optimistically biased, see above")
holdout.per_class.round(3)[["f1", "iou", "support"]]

## Fit on everything and persist

The final model is fitted on all labelled sites. Its recorded CV score comes from the
grouped cross-validation above, not from this fit — a model scored on its own training data
is meaningless.

The feature list and config hash are saved beside it, because a model applied with features
in a different order produces confident nonsense rather than an error.

In [ ]:
final_model = wh_train.make_model(BEST, cfg)
final_model.fit(table[features], table["class_id"])

model_path, manifest_path = wh_train.save_model(
    final_model, features, cfg, PARAMS, results[BEST], table
)
print(f"model    -> {model_path}")
print(f"manifest -> {manifest_path}")

In [ ]:
# Reload and confirm it round-trips.
reloaded, meta = wh_train.load_model(cfg)
print(f"{meta['model_class']}, {meta['n_features']} features, "
      f"CV macro F1 {meta['cv_macro_f1']:.3f}")
assert list(meta["feature_names"]) == list(features)
assert np.array_equal(reloaded.predict(table[features].head(100)),
                      final_model.predict(table[features].head(100)))
print("round-trip OK")

## Where is the model right and wrong, spatially?

Aggregate metrics say *which* classes are weak. These maps say **where** — and that is what
distinguishes a model problem from a label problem.

**Every prediction here comes from a model that has never seen the site being drawn.**
`fit_without_site` refits on the other 8 sites first. Predicting a site the model was
trained on would look excellent and tell you nothing.

Refitting and loading a site's 84-month stack are both expensive, so both are cached below —
inspecting more months of a site you have already looked at is nearly free.

In [ ]:
import wh_footprint

_model_cache = {}
_predictor_cache = {}

def held_out_view(site_id, year_month):
    """Predict one site-month with a model that never saw that site. Cached."""
    if site_id not in _model_cache:
        print(f"  fitting without site {site_id}...", flush=True)
        _model_cache[site_id] = wh_train.fit_without_site(table, cfg, BEST, site_id, features)
    if site_id not in _predictor_cache:
        print(f"  loading site {site_id} history...", flush=True)
        _predictor_cache[site_id] = wh_train.SitePredictor(manifest, cfg, PARAMS, site_id)

    tile, predicted, confidence = _predictor_cache[site_id].predict(
        _model_cache[site_id], features, year_month
    )
    mask_path = cfg.paths["labels"] / f"{Path(tile.path).stem}_labels.tif"
    manual = wh_tiles.read_mask(mask_path, tile.shape) if mask_path.exists() else None
    try:
        footprint = wh_footprint.load_mask(cfg, site_id)
    except FileNotFoundError:
        footprint = None
    return tile, predicted, confidence, manual, footprint


candidates = wh_train.labelled_tiles(cfg).sort_values(
    ["n_classes", "n_labelled"], ascending=False
)
print(f"{len(candidates)} hand-labelled tiles available")
candidates.head(10)[["site_id", "year_month", "n_classes", "n_labelled"]]

In [ ]:
INSPECT = [
    (candidates.iloc[0]["site_id"], candidates.iloc[0]["year_month"]),
    (candidates.iloc[1]["site_id"], candidates.iloc[1]["year_month"]),
]

for site_id, year_month in INSPECT:
    tile, predicted, confidence, manual, footprint = held_out_view(site_id, year_month)
    figure = wh_plots.plot_prediction_map(
        tile, predicted, cfg, manual, confidence, footprint, held_out=True
    )
    plt.show()

**Reading the agreement panel.** Green is correct, red is wrong, and unlabelled pixels are
left showing the imagery. Look for structure: errors concentrated in the basin interior mean
something different from errors scattered along class boundaries.

Boundary errors are largely mixed pixels and are expected. Errors filling a whole region —
especially at *high* confidence — mean the model has learned something wrong about that
surface, and are worth chasing.

### What each class gets mistaken for, in place

Splits the previous view by hand-labelled class: green where that class was got right, and
the colour of whatever it was confused with where it was not. This answers "my mud is being
called dry ground — everywhere, or only at the basin margin?", which the confusion matrix
cannot.

In [ ]:
site_id, year_month = INSPECT[0]
tile, predicted, _, manual, _ = held_out_view(site_id, year_month)

figure = wh_plots.plot_error_by_class_map(tile, predicted, manual, cfg)
plt.show()

### The same waterhole through time

One site across several months, from a model that has never seen it. Drying should show as
an orderly progression; if the predicted composition jumps around between adjacent months,
that is either a compositing artefact or the model being unstable — and the temporal
consistency pass in deliverable 7 is what will deal with it.

The site stack loads once here, so extra months are cheap.

In [ ]:
TIMESERIES_SITE = INSPECT[0][0]
N_MONTHS = 6

predictor = _predictor_cache[TIMESERIES_SITE]
months = predictor.months
chosen = [months[i] for i in np.linspace(0, len(months) - 1, N_MONTHS).round().astype(int)]

figure, axes = plt.subplots(2, len(chosen), figsize=(3.0 * len(chosen), 6.2),
                            constrained_layout=True)
for column, year_month in enumerate(chosen):
    tile, predicted, _, _, _ = held_out_view(TIMESERIES_SITE, year_month)
    axes[0, column].imshow(wh_plots.rgb_composite(tile))
    axes[0, column].set_title(year_month, fontsize=9)
    axes[1, column].imshow(wh_plots.rgb_composite(tile))
    axes[1, column].imshow(wh_plots.class_overlay(predicted, cfg, alpha=0.8),
                           interpolation="nearest")
    for row in (0, 1):
        axes[row, column].set_xticks([])
        axes[row, column].set_yticks([])

axes[0, 0].set_ylabel("RGB", fontsize=9)
axes[1, 0].set_ylabel("predicted", fontsize=9)
wh_plots.class_legend(axes[1, -1], cfg)
figure.suptitle(f"site {TIMESERIES_SITE} — model has NOT seen this site", fontsize=11)
plt.show()

## Where to spend the next labelling session

Ranked by what would most improve the model: classes at fewer than 3 sites cannot be
cross-validated at all, and sites with low per-site F1 are where generalisation is failing.

In [ ]:
print("classes needing more SITES (not more pixels):")
print(coverage[coverage["sites"] < 4][["class", "pixels", "sites"]].to_string(index=False))

print("\nworst-performing sites — look at these in the labeller and check the labels "
      "are right before blaming the model:")
print(evaluation.per_site.head(3).round(3).to_string())